In [54]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-5-5"

In [55]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)

    for block in message.content:
        if block.type == "text":
            return block.text

## 課題: ツール使用の往復を1回、通して動かす

- 対応: L22「ツール関数」、L23「ツールスキーマ」、L24「メッセージブロックの処理」、L25「ツール結果の送信」。
- 今回、新しく覚えること: **ツール使用の往復**(依頼を受け取る → 関数を実行する → 結果を返す)。
- 目標: 「今の時刻を HH:MM:SS 形式で教えてください」に対して、Claude に `get_current_datetime` ツールを呼ばせ、最終応答に実際の時刻を含めさせる。
- 流れ(L20・L25 の4ステップ): ①`tools` 付きで質問を送る → ②応答の中から ToolUse ブロックを探す → ③関数を実行して、`tool_result` を履歴に足す → ④もう一度送って、最終応答を得る。
- API 呼び出し: 2回(少額の料金がかかる)。TODO 1・2 は API なしで確認できる。
- ヒントは、下の「ヒント」のセルにある。まず、見ずに、試してみる。
- 注意: ページのコード `response.content[1]` は、「2番目のブロックが ToolUse」と決め打ちしている。この演習では、**種類(`block.type`)で探す**書き方にする(L24 の補足)。

In [56]:
# TODO 1: ツール関数 get_current_datetime を完成させる(L22)
#   - date_format が空(空文字など)なら、ValueError を発生させる。メッセージは、原因が分かるものにする。
#   - 空でなければ、datetime.now() を strftime(date_format) で文字列にして返す。
#   - 引数の既定値は "%Y-%m-%d %H:%M:%S"。
from datetime import datetime


def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format無しは受け付けられません")  # TODO: 原因が分かるメッセージにする
    return datetime.now().strftime(date_format)  # TODO: 現在時刻を date_format で整形して返す

In [57]:
# TODO 1 の動作確認(API は呼ばない)。すべて通れば、"OK" が表示される。
assert len(get_current_datetime("%H:%M")) == 5
assert len(get_current_datetime()) == 19

try:
    get_current_datetime("")
    raise AssertionError("空の date_format で、ValueError が出ていない")
except ValueError as e:
    print("エラーメッセージ:", e)

print("OK")

エラーメッセージ: date_format無しは受け付けられません
OK


In [58]:
# TODO 2: ツールスキーマ(L23)を完成させる
#   - name: 関数名と同じにする。
#   - description: 「何をするか」「いつ使うか」「何を返すか」を入れる(2〜3文。日本語でも英語でもよい)。
#   - input_schema: date_format(文字列)を持つ object。date_format は省略可能なので、required は空のリスト。
#   - date_format の description も書く(strftime の書式であること)。
get_current_datetime_schema = {
    "name": "get_current_datetime",  # TODO: 関数名と同じ
    "description": "現在時刻を取得します。時間のフォーマットを指定することができます。",  # TODO: 何をするか・いつ使うか・何を返すか
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",  # TODO: 確認(文字列)
                "description": "現在時刻です。Pythonのstrftimeフォーマットです。",  # TODO: strftime の書式であることを説明
            }
        },
        "required": [],
    },
}

In [59]:
# TODO 2 の動作確認(API は呼ばない)。
assert get_current_datetime_schema["name"] == "get_current_datetime"
assert get_current_datetime_schema["description"] != "TODO"
assert get_current_datetime_schema["input_schema"]["type"] == "object"
assert get_current_datetime_schema["input_schema"]["required"] == []
print("OK")

OK


In [60]:
# TODO 3: tools 付きで最初の呼び出しをして、ToolUse ブロックを探す(L24)
#   - messages に、ユーザーメッセージを足す。
#   - client.messages.create(...) に tools=[get_current_datetime_schema] を渡す。
#   - 応答の stop_reason と、content の各ブロックの type を表示する。
#   - content を for で回して、type が "tool_use" のブロックを見つける(位置を決め打ちしない)。
messages = []
add_user_message(messages, "今の時刻を HH:MM:SS 形式で教えてください。")

response = client.messages.create(
    model=model,
    max_tokens=4000,
    messages=messages,
    tools=[get_current_datetime_schema] # TODO: tools を渡す
)

print(response.stop_reason, [block.type for block in response.content])

tool_use_block = None
for block in response.content:
    if block.type == "tool_use":  # TODO: ToolUse ブロックの type 名
        tool_use_block = block

print(tool_use_block)

tool_use ['tool_use']
ToolUseBlock(id='toolu_01AGc8e2BPXH3iWenbW1vF68', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)


In [61]:
# TODO 4: 関数を実行して、tool_result を履歴に足す(L25)
#   - ToolUse ブロックの input(辞書)を、** で展開して関数に渡す。
#   - 履歴に、①アシスタントメッセージ(response.content をそのまま)②ユーザーメッセージ(tool_result)を、この順で足す。
#   - tool_result の tool_use_id は、ToolUse ブロックの id と一致させる。content は文字列にする。
tool_output = get_current_datetime(**tool_use_block.input)  # TODO: 確認(** で展開)
print("関数の出力:", tool_output)

messages.append({"role": "assistant", "content": response.content})  # TODO: 確認

messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": tool_use_block.id,  # TODO: ToolUse ブロックの id
        "content": tool_output,  # TODO: 関数の出力(文字列)
        "is_error": False,
    }],
})

関数の出力: 16:19:25


In [62]:
# TODO 5: 最終リクエストを送り、最終応答を表示する(L25)
#   - ツールを呼ばせる気がなくても、tools を含める(ページの記述)。
#   - 応答から、type が "text" のブロックの本文を取り出して表示する。
final_response = client.messages.create(
    model=model,
    max_tokens=4000,
    messages=messages,
    tools=[get_current_datetime_schema] # TODO: tools を渡す
)

print(final_response.stop_reason, [block.type for block in final_response.content])
for block in final_response.content:
    if block.type == "text":
        print(block.text)

end_turn ['text']
現在の時刻は **16:19:25** です。


## ヒント(困ったときだけ開く)

**TODO 1(ツール関数)**
- エラーメッセージの例: `"date_format cannot be empty"`(日本語でもよい。何が悪いかが分かること)。
- 整形: `datetime.now().strftime(date_format)`。
- 確認のセルで、どの `assert` で止まったかを見る。

**TODO 2(スキーマ)**
- name は `"get_current_datetime"`。
- description の例: 「現在の日付と時刻を、指定した書式の文字列で返す。利用者が現在の時刻や日付を尋ねたときに使う。」
- date_format の type は `"string"`。description の例: 「返す日時の書式。Python の strftime の書式コードを使う。」

**TODO 3(最初の呼び出し)**
- `tools=[get_current_datetime_schema]` を、`create(...)` の引数に足す。
- ToolUse ブロックの type は `"tool_use"`。
- 応答に、テキストブロックと ToolUse ブロックの両方が入るかを、`print` の1行目で見る。

**TODO 4(結果を足す)**
- `tool_use_id` は `tool_use_block.id`。
- `content` は `tool_output`(関数の出力。すでに文字列)。
- 履歴に足す順序を、間違えない: アシスタント → ユーザー。

**TODO 5(最終リクエスト)**
- TODO 3 と同じく、`tools=[get_current_datetime_schema]`。
- 最終応答の本文に、関数の出力の時刻が入っているかを、目で見て確かめる。

**エラーが出たら**
- 最後の行を読む。`AttributeError: 'NoneType' object has no attribute 'input'` は、`tool_use_block` が見つかっていない、という意味のことが多い。TODO 3 の type 名と、`print(tool_use_block)` の結果を確認する。
- `400` が出たら、エラーメッセージの全文を読む(どのブロックのどの規則に違反しているかが書かれている)。

## 振り返り(自分の言葉で書く。短くてよい)

1. 最初の応答で、`stop_reason` は何で、`content` にはどんなブロックが入っていたか。
2. Claude が関数に渡した引数(`tool_use_block.input`)は、何だったか。依頼した形式(HH:MM:SS)と合っていたか。
3. (任意の実験)TODO 5 で `tools` を外して実行すると、どうなるか。エラーメッセージを読んで、何と書かれていたかを書く。
4. `tool_result` を入れるメッセージの `role` が `user` なのは、なぜか。